In [9]:
import pandas as pd
import numpy as np

np.random.seed(42)

data = {
    "Time": np.random.randint(0, 100000, 10000),
    "Amount": np.random.uniform(0, 5000, 10000),
    "Class": np.random.choice([0, 1], size=10000, p=[0.98, 0.02])
}

for i in range(1, 11):
    data[f"V{i}"] = np.random.randn(10000)

df = pd.DataFrame(data)

print(df.head())
print(df["Class"].value_counts())

    Time       Amount  Class        V1        V2        V3        V4  \
0  15795  3805.029915      0 -1.060732 -1.232185  0.107279  0.950853   
1    860  3565.071459      0  1.474460  0.679117  0.845818  1.235588   
2  76820  1831.431931      0  0.396271 -0.888997  0.463978  0.454541   
3  54886   264.679678      0  0.622664 -1.122806 -0.584869 -0.398743   
4   6265  3774.662699      0 -0.329205 -0.603839  0.299645 -0.300218   

         V5        V6        V7        V8        V9       V10  
0  0.427815 -0.338338 -1.603431  0.340745  0.290741 -0.085792  
1 -1.104387 -0.033446 -1.758243  0.201586  0.256928 -0.682929  
2 -0.652797 -0.184911  0.044498  0.733152  2.406181  1.338748  
3 -1.343019 -1.021507 -0.084406  0.272659 -0.594298 -0.360510  
4  0.931582 -0.028713  0.647545 -0.046998  0.374438  0.579186  
Class
0    9785
1     215
Name: count, dtype: int64


In [10]:
from imblearn.over_sampling import SMOTE

X = df.drop("Class", axis=1)
y = df["Class"]

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print(y_resampled.value_counts())

Class
0    9785
1    9785
Name: count, dtype: int64


In [11]:
pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# original data
X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("WITHOUT SMOTE:\n")
print(classification_report(y_test, y_pred))

WITHOUT SMOTE:

              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1956
           1       0.00      0.00      0.00        44

    accuracy                           0.98      2000
   macro avg       0.49      0.50      0.49      2000
weighted avg       0.96      0.98      0.97      2000



c:\Users\Vanshika\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Vanshika\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Vanshika\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _war

In [13]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

model_s = RandomForestClassifier()
model_s.fit(X_train_s, y_train_s)

y_pred_s = model_s.predict(X_test_s)

print("WITH SMOTE:\n")
print(classification_report(y_test_s, y_pred_s))

WITH SMOTE:

              precision    recall  f1-score   support

           0       0.99      0.96      0.97      1965
           1       0.96      0.99      0.98      1949

    accuracy                           0.97      3914
   macro avg       0.98      0.98      0.97      3914
weighted avg       0.98      0.97      0.97      3914



In [14]:
import pickle

pickle.dump(model_s, open("fraud_model.pkl", "wb"))

In [17]:
import pickle

with open("fraud_model.pkl", "wb") as f:
    pickle.dump(model_s, f)

In [18]:
import os
print(os.listdir())

['app.py', 'data', 'fraud_model.pkl', 'model.pkl', 'notebook.ipynb', 'requirements.txt']


In [19]:
import os
print(os.path.expanduser("~"))

C:\Users\Vanshika


In [22]:
import streamlit as st
import pickle
import numpy as np

# load model
model = pickle.load(open("fraud_model.pkl", "rb"))

st.title("💳 Credit Card Fraud Detection")

st.write("Enter transaction details:")

# inputs
time = st.number_input("Transaction Time")
amount = st.number_input("Transaction Amount")

v_inputs = []
for i in range(1, 11):
    val = st.number_input(f"V{i}")
    v_inputs.append(val)

# prediction
if st.button("Predict"):
    features = np.array([[time, amount] + v_inputs])
    
    prediction = model.predict(features)
    
    if prediction[0] == 1:
        st.error("🚨 Fraud Transaction Detected!")
    else:
        st.success("✅ Safe Transaction")

2026-04-12 11:04:25.029 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-12 11:04:25.293 
  command:

    streamlit run C:\Users\Vanshika\AppData\Roaming\Python\Python314\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-04-12 11:04:25.295 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-12 11:04:25.296 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-12 11:04:25.298 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-12 11:04:25.299 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-12 11:04:25.300 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-12 11:04:25.302 T

In [1]:
import os
print(os.getcwd())

d:\datascienece_journey\Credit Card Fraud Detection with Machine Learning & Deployment
